In [5]:
"""
Generate a daily calendar/holiday/events feature table for 2026-03-01 to 2026-07-17.
- Federal holidays: computed (no download needed)
- NYC school dates: hardcoded from the verified 2025-2026 DOE calendar
"""

from datetime import date, timedelta

import polars as pl
from pandas.tseries.holiday import USFederalHolidayCalendar

START_DATE = "2026-03-01"
END_DATE = "2026-07-17"

# verified against schools.nyc.gov / NYC DOE 2025-2026 calendar
NYC_SPRING_RECESS_START = "2026-04-02"
NYC_SPRING_RECESS_END = "2026-04-10"
NYC_LAST_DAY_OF_SCHOOL = "2026-06-26"

NYC_SINGLE_DAY_SCHOOL_CLOSURES = [
    "2026-03-20",  # Eid al-Fitr
    "2026-05-25",  # Memorial Day
    "2026-05-27",  # Eid al-Adha
    "2026-06-04",  # Anniversary Day
    "2026-06-19",  # Juneteenth
]



# ---------------------------------------------------------------------------
# Major Manhattan events, manually verified via web search (more reliable for
# named recurring events than the NYPD permits API, which is better suited to
# the long tail of smaller filming/street-fair closures). Each has a rough
# lat/lon centroid of its route/area for a "nearby route" distance check.
# ---------------------------------------------------------------------------
MAJOR_MANHATTAN_EVENTS = [
    {
        "event_name": "St. Patrick's Day Parade",
        "event_date": "2026-03-17",
        "start_time": "11:00",
        "end_time": "16:30",
        "borough": "Manhattan",
        "event_type": "Parade",
        "area_desc": "5th Ave, 44th St to 79th St",
        "lat": 40.7647, "lon": -73.9720,  # midpoint of route
    },
    {
        "event_name": "Easter Parade",
        "event_date": "2026-04-05",
        "start_time": "10:00",
        "end_time": "16:00",
        "borough": "Manhattan",
        "event_type": "Parade",
        "area_desc": "5th Ave, 49th St to 57th St",
        "lat": 40.7614, "lon": -73.9776,
    },
    {
        "event_name": "Puerto Rican Day Parade",
        "event_date": "2026-06-14",
        "start_time": "11:00",
        "end_time": "17:00",
        "borough": "Manhattan",
        "event_type": "Parade",
        "area_desc": "5th Ave, 44th St to 86th St",
        "lat": 40.7724, "lon": -73.9640,
    },
    {
        "event_name": "NYC Pride March",
        "event_date": "2026-06-28",
        "start_time": "12:00",
        "end_time": "18:00",
        "borough": "Manhattan",
        "event_type": "Parade",
        "area_desc": "5th Ave (25th St) to Christopher St, Greenwich Village",
        "lat": 40.7390, "lon": -73.9970,
    },
    {
        "event_name": "Fleet Week / SAIL250",
        "event_date": "2026-07-03",
        "start_time": "08:00",
        "end_time": "20:00",
        "borough": "Manhattan",
        "event_type": "Multi-day maritime event",
        "area_desc": "Hudson River piers / Manhattan waterfront (through July 8)",
        "lat": 40.7654, "lon": -74.0021,
    },
    {
        "event_name": "Fleet Week / SAIL250",
        "event_date": "2026-07-04",
        "start_time": "08:00",
        "end_time": "22:00",
        "borough": "Manhattan",
        "event_type": "Multi-day maritime event",
        "area_desc": "Hudson River piers / Manhattan waterfront (through July 8)",
        "lat": 40.7654, "lon": -74.0021,
    },
    {
        "event_name": "Macy's 4th of July Fireworks",
        "event_date": "2026-07-04",
        "start_time": "21:25",
        "end_time": "22:00",
        "borough": "Manhattan",
        "event_type": "Fireworks / large public gathering",
        "area_desc": "East River / Hudson River viewing areas",
        "lat": 40.7484, "lon": -73.9705,
    },
    {
        "event_name": "Fleet Week / SAIL250",
        "event_date": "2026-07-05",
        "start_time": "08:00",
        "end_time": "20:00",
        "borough": "Manhattan",
        "event_type": "Multi-day maritime event",
        "area_desc": "Hudson River piers / Manhattan waterfront (through July 8)",
        "lat": 40.7654, "lon": -74.0021,
    },
    {
        "event_name": "Fleet Week / SAIL250",
        "event_date": "2026-07-06",
        "start_time": "08:00",
        "end_time": "20:00",
        "borough": "Manhattan",
        "event_type": "Multi-day maritime event",
        "area_desc": "Hudson River piers / Manhattan waterfront (through July 8)",
        "lat": 40.7654, "lon": -74.0021,
    },
    {
        "event_name": "Fleet Week / SAIL250",
        "event_date": "2026-07-07",
        "start_time": "08:00",
        "end_time": "20:00",
        "borough": "Manhattan",
        "event_type": "Multi-day maritime event",
        "area_desc": "Hudson River piers / Manhattan waterfront (through July 8)",
        "lat": 40.7654, "lon": -74.0021,
    },
    {
        "event_name": "Fleet Week / SAIL250",
        "event_date": "2026-07-08",
        "start_time": "08:00",
        "end_time": "20:00",
        "borough": "Manhattan",
        "event_type": "Multi-day maritime event",
        "area_desc": "Hudson River piers / Manhattan waterfront (through July 8)",
        "lat": 40.7654, "lon": -74.0021,
    },
    # Summer Streets typically runs in August — outside the 03/01-07/17 window.
    # Add here with verified dates if your study period is extended later.
]


def get_major_events_table() -> pl.DataFrame:
    """Returns the manually-verified major events as a DataFrame — one row
    per event-day, matching the requested feature schema."""
    return pl.DataFrame(MAJOR_MANHATTAN_EVENTS)


def _haversine_m(lat1, lon1, lat2, lon2):
    import math
    R = 6371000
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = math.sin(dphi / 2) ** 2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda / 2) ** 2
    return 2 * R * math.asin(math.sqrt(a))


def add_event_nearby_flag(
    stops_df: pl.DataFrame,
    events_df: pl.DataFrame,
    stop_lat_col: str = "stop_lat",
    stop_lon_col: str = "stop_lon",
    threshold_m: float = 800.0,
) -> pl.DataFrame:
    """
    For a table with one row per (stop or route-segment), adds an
    Event_Nearby flag if any major event's date matches service_date AND
    the stop is within `threshold_m` meters of that event's area centroid.

    stops_df must have: service_date, stop_lat, stop_lon
    """
    events_by_date = {
        row["event_date"]: (row["lat"], row["lon"], row["event_name"])
        for row in events_df.iter_rows(named=True)
    }

    def _check(row):
        d = row["service_date"]
        if d not in events_by_date:
            return False
        elat, elon, _ = events_by_date[d]
        dist = _haversine_m(row[stop_lat_col], row[stop_lon_col], elat, elon)
        return dist <= threshold_m

    return stops_df.with_columns(
        pl.struct(["service_date", stop_lat_col, stop_lon_col])
        .map_elements(_check, return_dtype=pl.Boolean)
        .alias("Event_Nearby")
    )





def build_calendar_dataset(start_date: str, end_date: str, include_events: bool = True) -> pl.DataFrame:
    d0 = date.fromisoformat(start_date)
    d1 = date.fromisoformat(end_date)
    all_dates = [(d0 + timedelta(days=i)).isoformat() for i in range((d1 - d0).days + 1)]

    df = pl.DataFrame({"service_date": all_dates})
    df = df.with_columns(
        pl.col("service_date").str.strptime(pl.Date, "%Y-%m-%d").alias("date_obj")
    )

    # day of week / weekend
    df = df.with_columns([
        pl.col("date_obj").dt.weekday().alias("day_of_week"),  # 1=Mon .. 7=Sun
        (pl.col("date_obj").dt.weekday() >= 6).alias("is_weekend"),
    ])

    # US federal holidays (no download — computed)
    cal = USFederalHolidayCalendar()
    fed_holidays = cal.holidays(start=start_date, end=end_date)
    fed_holiday_strs = set(d.strftime("%Y-%m-%d") for d in fed_holidays)
    df = df.with_columns(
        pl.col("service_date").is_in(list(fed_holiday_strs)).alias("is_federal_holiday")
    )

    # NYC school calendar
    df = df.with_columns([
        (
            (pl.col("service_date") >= NYC_SPRING_RECESS_START)
            & (pl.col("service_date") <= NYC_SPRING_RECESS_END)
        ).alias("is_spring_recess"),
        (pl.col("service_date") > NYC_LAST_DAY_OF_SCHOOL).alias("is_summer_break"),
        pl.col("service_date")
        .is_in(NYC_SINGLE_DAY_SCHOOL_CLOSURES)
        .alias("is_single_day_school_closure"),
    ])

    df = df.with_columns(
        (
            (~pl.col("is_weekend"))
            & (~pl.col("is_spring_recess"))
            & (~pl.col("is_summer_break"))
            & (~pl.col("is_single_day_school_closure"))
            & (~pl.col("is_federal_holiday"))
        ).alias("is_school_day")
    )

    df = df.drop("date_obj")

    if include_events:
        major_events = get_major_events_table()

        major_daily = (
            major_events.group_by("event_date")
            .agg(pl.len().alias("major_event_count"))
            .rename({"event_date": "service_date"})
        )

        df = (
            df.join(major_daily, on="service_date", how="left")
            .with_columns([
                pl.col("major_event_count").fill_null(0),
                (pl.col("major_event_count") > 0).alias("has_any_event"),
            ])
        )

    return df


if __name__ == "__main__":
    print("=== Major Manhattan Events (verified) ===")
    major_events_df = get_major_events_table()
    print(major_events_df)
    major_events_df.write_csv("major_manhattan_events.csv")

    print("\n=== Daily Calendar Dataset ===")
    calendar_df = build_calendar_dataset(START_DATE, END_DATE, include_events=True)
    print(calendar_df)
    calendar_df.write_parquet("calendar_dataset.parquet")
    calendar_df.write_csv("calendar_dataset.csv")
    print(f"\nWrote {calendar_df.shape[0]} rows to calendar_dataset.parquet / .csv")
    print(f"Wrote {major_events_df.shape[0]} rows to major_manhattan_events.csv")

=== Major Manhattan Events (verified) ===
shape: (11, 9)
┌────────────┬────────────┬────────────┬──────────┬───┬───────────┬───────────┬─────────┬──────────┐
│ event_name ┆ event_date ┆ start_time ┆ end_time ┆ … ┆ event_typ ┆ area_desc ┆ lat     ┆ lon      │
│ ---        ┆ ---        ┆ ---        ┆ ---      ┆   ┆ e         ┆ ---       ┆ ---     ┆ ---      │
│ str        ┆ str        ┆ str        ┆ str      ┆   ┆ ---       ┆ str       ┆ f64     ┆ f64      │
│            ┆            ┆            ┆          ┆   ┆ str       ┆           ┆         ┆          │
╞════════════╪════════════╪════════════╪══════════╪═══╪═══════════╪═══════════╪═════════╪══════════╡
│ St.        ┆ 2026-03-17 ┆ 11:00      ┆ 16:30    ┆ … ┆ Parade    ┆ 5th Ave,  ┆ 40.7647 ┆ -73.972  │
│ Patrick's  ┆            ┆            ┆          ┆   ┆           ┆ 44th St   ┆         ┆          │
│ Day Parade ┆            ┆            ┆          ┆   ┆           ┆ to 79th   ┆         ┆          │
│            ┆            ┆       

In [6]:
calendar_df.null_count()

service_date,day_of_week,is_weekend,is_federal_holiday,is_spring_recess,is_summer_break,is_single_day_school_closure,is_school_day,major_event_count,has_any_event
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,0,129


In [7]:
calendar_df.shape

(139, 10)